## Task 1: Data Cleaning:
Identify anomalies in the Vict Age (Victim Age) column, 
such as negative values or Valid age range: 0–120 (inclusive). Exclude null/blank 
and values outside this range. 
Calculate the average victim age for each AREA NAME while strictly excluding 
these noisy data points/outliers.

### Pandas version

In [54]:
import pandas as pd
import sqlite3

In [55]:
#drop values with all Nans
df = pd.read_csv("Crime_Data_from_2020_to_Present.csv").dropna(how = "all") 

In [56]:
df.columns.tolist() #check every attribute of the data set

['DR_NO',
 'Date Rptd',
 'DATE OCC',
 'TIME OCC',
 'AREA',
 'AREA NAME',
 'Rpt Dist No',
 'Part 1-2',
 'Crm Cd',
 'Crm Cd Desc',
 'Mocodes',
 'Vict Age',
 'Vict Sex',
 'Vict Descent',
 'Premis Cd',
 'Premis Desc',
 'Weapon Used Cd',
 'Weapon Desc',
 'Status',
 'Status Desc',
 'Crm Cd 1',
 'Crm Cd 2',
 'Crm Cd 3',
 'Crm Cd 4',
 'LOCATION',
 'Cross Street',
 'LAT',
 'LON']

In [57]:
# Take the "Vict Age" column and try to turn every value into a number.
# If something cannot be turned into a number, make it NaN (missing value).
vict_age_num = pd.to_numeric(df["Vict Age"], errors="coerce")

# Keep only the rows where:
# 1. the age is NOT missing
# 2. the age is between 0 and 120
# .copy() makes a new separate DataFrame
clean_age_df = df[vict_age_num.notna() & vict_age_num.between(0, 120)].copy()

# Replace the old "Vict Age" column in the cleaned DataFrame
# with the cleaned numeric ages from vict_age_num
clean_age_df["Vict Age"] = vict_age_num[vict_age_num.notna() & vict_age_num.between(0, 120)]

# Start building the final result:
task1_pandas = (
    clean_age_df
    .groupby("AREA NAME", as_index=False)["Vict Age"]  # Group rows by "AREA NAME"
    .mean()  # Find the average of "Vict Age" for each area
    .rename(columns={"Vict Age": "avg_victim_age"}) # Rename the column from "Vict Age" to "avg_victim_age"
    .sort_values("avg_victim_age", ascending=False)  # Sort from biggest average age to smallest average age
)

task1_pandas.head(10)

,AREA NAME,avg_victim_age
16,Topanga,34.385990
17,Van Nuys,31.892481
18,West LA,31.217313
19,West Valley,31.191734
3,Foothill,29.922822
2,Devonshire,29.723814
15,Southwest,29.710846
20,Wilshire,29.516401
12,Pacific,29.139376
7,Mission,29.119703


### SQL version 

In [58]:
# make sql table
conn = sqlite3.connect(":memory:") #create a temporary database in sql

# to_sql() copies your Pandas table into SQL as a table named crime_data
df.to_sql("crime_data", conn, if_exists="replace", index=False) 


1004991

In [59]:
# Create a multi-line SQL query and store it in a Python variable called query1
query1 = """
SELECT
    "AREA NAME",
    ROUND(AVG(CAST("Vict Age" AS REAL)), 2) AS avg_victim_age
FROM crime_data
WHERE "Vict Age" IS NOT NULL
  AND CAST("Vict Age" AS INTEGER) BETWEEN 0 AND 120
GROUP BY "AREA NAME"
ORDER BY avg_victim_age DESC;
"""

# Run the SQL query using the database connection named conn
# Then save the result as a pandas DataFrame called task1_sql
task1_sql = pd.read_sql_query(query1, conn)

# Show the first 10 rows of the SQL result
task1_sql.head(10)

,AREA NAME,avg_victim_age
0,Topanga,34.39
1,Van Nuys,31.89
2,West LA,31.22
3,West Valley,31.19
4,Foothill,29.92
5,Devonshire,29.72
6,Southwest,29.71
7,Wilshire,29.52
8,Pacific,29.14
9,Mission,29.12


# Task 2

Filter incidents where the crime description contains the keywords 
"WEAPON" OR "ASSAULT" (case-insensitive) using partial string matching. 
Requirements: Perform a keyword search within the crime description field. You 
must account for case sensitivity and partial matches. 

## Pandas:

In [60]:
# Create a new DataFrame called task2_pandas
task2_pandas = df[
   
    # Look at the "Crm Cd Desc" column
    # If a value is empty (NaN), replace it with an empty string ""
    # Then search for the words "WEAPON" or "ASSAULT"
    # case=False means ignore uppercase/lowercase differences
    # regex=True means treat "WEAPON|ASSAULT" as a pattern:
    # the | means OR, so it finds rows containing either word
    df["Crm Cd Desc"]
    .fillna("")
    .str.contains("WEAPON|ASSAULT", case=False, regex=True)

# After finding matching rows, keep only these columns    
][["DR_NO", "AREA NAME", "Crm Cd Desc", "DATE OCC", "TIME OCC"]]

# Show the first 10 rows of the result
task2_pandas.head(10)

,DR_NO,AREA NAME,Crm Cd Desc,DATE OCC,TIME OCC
1,201516622,N Hollywood,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",10/18/2020 0:00,1845
16,201820230,Southeast,INTIMATE PARTNER - SIMPLE ASSAULT,11/8/2020 0:00,730
17,201707577,Devonshire,BATTERY - SIMPLE ASSAULT,3/20/2020 0:00,1320
24,201608553,Foothill,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",4/28/2020 0:00,1730
29,202013248,Olympic,INTIMATE PARTNER - SIMPLE ASSAULT,8/9/2020 0:00,2250
32,201809657,Southeast,INTIMATE PARTNER - SIMPLE ASSAULT,4/15/2020 0:00,1550
35,201911106,Mission,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",4/26/2020 0:00,1908
36,201819475,Southeast,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",10/24/2020 0:00,2250
37,202106474,Topanga,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",2/24/2020 0:00,1835
47,201810632,Southeast,BATTERY - SIMPLE ASSAULT,5/4/2020 0:00,1530


## SQL Version: 

In [61]:
# Store the SQL query inside a Python variable called query2
query2 = """
SELECT
    "DR_NO",         -- Get the DR_NO column
    "AREA NAME",     -- Get the AREA NAME column
    "Crm Cd Desc",   -- Get the crime description column
    "DATE OCC",      -- Get the date the crime happened
    "TIME OCC"       -- Get the time the crime happened
FROM crime_data      -- Use the table called crime_data
WHERE UPPER(COALESCE("Crm Cd Desc", "")) LIKE '%WEAPON%'   -- Find rows with WEAPON
   OR UPPER(COALESCE("Crm Cd Desc", "")) LIKE '%ASSAULT%'; -- Or find rows with ASSAULT
"""

# Run the SQL query and save the result as a pandas DataFrame
task2_sql = pd.read_sql_query(query2, conn)

# Show the first 10 rows
task2_sql.head(10)

,DR_NO,AREA NAME,Crm Cd Desc,DATE OCC,TIME OCC
0,201516622,N Hollywood,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",10/18/2020 0:00,1845
1,201820230,Southeast,INTIMATE PARTNER - SIMPLE ASSAULT,11/8/2020 0:00,730
2,201707577,Devonshire,BATTERY - SIMPLE ASSAULT,3/20/2020 0:00,1320
3,201608553,Foothill,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",4/28/2020 0:00,1730
4,202013248,Olympic,INTIMATE PARTNER - SIMPLE ASSAULT,8/9/2020 0:00,2250
5,201809657,Southeast,INTIMATE PARTNER - SIMPLE ASSAULT,4/15/2020 0:00,1550
6,201911106,Mission,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",4/26/2020 0:00,1908
7,201819475,Southeast,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",10/24/2020 0:00,2250
8,202106474,Topanga,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",2/24/2020 0:00,1835
9,201810632,Southeast,BATTERY - SIMPLE ASSAULT,5/4/2020 0:00,1530


# Task 3:
For each administrative district (AREA NAME), identify the three most 
frequent types of crime. Requirements: Aggregate data by area and crime type, 
then rank them to extract the top 3 for each location. 

## Pandas Version

In [62]:
# Group the data by AREA NAME and Crm Cd Desc
# Then count how many times each crime appears in each area
crime_counts = (
    df.groupby(["AREA NAME", "Crm Cd Desc"], as_index=False)
    .size()
    .rename(columns={"size": "crime_count"})
)

# Create a rank column inside each AREA NAME group
# Highest crime_count gets rank 1
crime_counts["rank"] = (
    crime_counts.groupby("AREA NAME")["crime_count"]
    .rank(method="first", ascending=False)
)

# Keep only the top 3 crimes in each area
# Then sort the results by area, rank, and crime description
task3_pandas = (
    crime_counts[crime_counts["rank"] <= 3]
    .sort_values(["AREA NAME", "rank", "Crm Cd Desc"])
)

# Show the first 20 rows
task3_pandas.head(20)

,AREA NAME,Crm Cd Desc,crime_count,rank
113,77th Street,VEHICLE - STOLEN,8767,1.0
2,77th Street,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",5921,2.0
4,77th Street,BATTERY - SIMPLE ASSAULT,4707,3.0
137,Central,BURGLARY FROM VEHICLE,9695,1.0
123,Central,BATTERY - SIMPLE ASSAULT,6806,2.0
231,Central,VEHICLE - STOLEN,5050,3.0
343,Devonshire,VEHICLE - STOLEN,3959,1.0
252,Devonshire,BURGLARY,3552,2.0
327,Devonshire,THEFT OF IDENTITY,3537,3.0
451,Foothill,VEHICLE - STOLEN,4539,1.0


## SQL Version

In [63]:
# Create a SQL query and store it in a Python variable called query3
query3 = """
-- create a temporary table called crime_counts
WITH crime_counts AS (
    SELECT
        "AREA NAME",              -- Get the area name
        "Crm Cd Desc",            -- Get the crime description
        COUNT(*) AS crime_count   -- Count how many times each crime appears in each area
    FROM crime_data
    GROUP BY "AREA NAME", "Crm Cd Desc"   -- Group by area and crime description
),

-- create another temporary table called ranked
ranked AS (
    SELECT
        "AREA NAME",              -- Keep the area name
        "Crm Cd Desc",            -- Keep the crime description
        crime_count,              -- Keep the crime count
        ROW_NUMBER() OVER (       -- Give each row a rank number
            PARTITION BY "AREA NAME"   -- Restart ranking for each area
            ORDER BY crime_count DESC, "Crm Cd Desc" ASC   -- Rank highest crime count first
        ) AS rank_num
    FROM crime_counts
)

-- Show the final result
SELECT *
FROM ranked
WHERE rank_num <= 3              -- Keep only the top 3 crimes in each area
ORDER BY "AREA NAME", rank_num;  -- Sort by area name and rank
"""

# Run the SQL query and save the result as a pandas DataFrame
task3_sql = pd.read_sql_query(query3, conn)

# Show the first 20 rows
task3_sql.head(20)

,AREA NAME,Crm Cd Desc,crime_count,rank_num
0,77th Street,VEHICLE - STOLEN,8767,1
1,77th Street,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",5921,2
2,77th Street,BATTERY - SIMPLE ASSAULT,4707,3
3,Central,BURGLARY FROM VEHICLE,9695,1
4,Central,BATTERY - SIMPLE ASSAULT,6806,2
5,Central,VEHICLE - STOLEN,5050,3
6,Devonshire,VEHICLE - STOLEN,3959,1
7,Devonshire,BURGLARY,3552,2
8,Devonshire,THEFT OF IDENTITY,3537,3
9,Foothill,VEHICLE - STOLEN,4539,1


# Task 4: 
Identify the "high-risk hours" of the day by determining when crime 
incidents occur most frequently in Los Angeles. 

Requirements: Extract the hour from the timestamp and analyze the hourly distribution of crime records. Use 
TIME OCC (in HHMM format) to compute hour 

## Pandas

In [64]:
# Take the "TIME OCC" column
# Replace missing values with 0
# Turn the values into integers, then strings
# Add leading zeros so every time has 4 digits like 0030, 0915, 1345
time_str = df["TIME OCC"].fillna(0).astype(int).astype(str).str.zfill(4)

# Make a copy of the original DataFrame
df_time = df.copy()

# Create a new column called "hour"
# Take the first 2 digits from the time string and turn them into an integer
df_time["hour"] = time_str.str[:2].astype(int)

# Group the data by hour
# Count how many incidents happened in each hour
# Rename the count column to incident_count
# Sort from highest incident count to lowest
task4_pandas = (
    df_time.groupby("hour", as_index=False)
    .size()
    .rename(columns={"size": "incident_count"})
    .sort_values("incident_count", ascending=False)
)

# Show the first 10 rows
task4_pandas.head(10)

,hour,incident_count
12,12,67813
18,18,59958
17,17,58811
20,20,56350
19,19,55597
16,16,52976
15,15,52824
21,21,50793
14,14,49301
22,22,49103


# SQL Version

In [65]:
# Create a SQL query and store it in a Python variable called query4
query4 = """
SELECT
    -- Take TIME OCC, replace missing values with 0,
    -- turn it into a 4-digit string like 0030 or 1345,
    -- take the first 2 digits, and turn that into the hour
    CAST(SUBSTR(printf('%04d', CAST(COALESCE("TIME OCC", 0) AS INTEGER)), 1, 2) AS INTEGER) AS hour,

    -- Count how many incidents happened in each hour
    COUNT(*) AS incident_count

-- Use the crime_data table
FROM crime_data

-- Group rows by hour
GROUP BY hour

-- Show the hours with the most incidents first
ORDER BY incident_count DESC;
"""

# Run the SQL query and save the result as a pandas DataFrame
task4_sql = pd.read_sql_query(query4, conn)

# Show the result
task4_sql

,hour,incident_count
0,12,67813
1,18,59958
2,17,58811
3,20,56350
4,19,55597
5,16,52976
6,15,52824
7,21,50793
8,14,49301
9,22,49103


In [5]:
import pandas as pd
df = pd.read_csv("Adidas_Global.csv")
df.head(10)

C:\Users\JUAND\AppData\Local\Temp\ipykernel_21000\247813120.py:2: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Adidas_Global.csv")


,snapshot_date,country_code,product_name,model_number,currency,price_local,sale_price_local,gender_segment,size_label,category,...,rating_count,badge_texts,product_url,canonical_url,image_url,best_for_ids,seen_market_count,seen_markets,market_record_source,size_record_source
0,2026-03-09,BR,AGASALHO COM CAPUZ BIG LOGO,KSG46,BRL,429.99,549.99,M,2GG,Lifestyle,...,-99.0,NaN,https://www.adidas.com.br/agasalho-com-capuz-b...,https://www.adidas.com.br/agasalho-com-capuz-b...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,BR,api,api_exact
1,2026-03-09,BR,AGASALHO COM CAPUZ BIG LOGO,KSG46,BRL,429.99,549.99,M,2GG/T,Lifestyle,...,-99.0,NaN,https://www.adidas.com.br/agasalho-com-capuz-b...,https://www.adidas.com.br/agasalho-com-capuz-b...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,BR,api,api_exact
2,2026-03-09,BR,AGASALHO COM CAPUZ BIG LOGO,KSG46,BRL,429.99,549.99,M,2XT2,Lifestyle,...,-99.0,NaN,https://www.adidas.com.br/agasalho-com-capuz-b...,https://www.adidas.com.br/agasalho-com-capuz-b...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,BR,api,api_exact
3,2026-03-09,BR,AGASALHO COM CAPUZ BIG LOGO,KSG46,BRL,429.99,549.99,M,2XT3,Lifestyle,...,-99.0,NaN,https://www.adidas.com.br/agasalho-com-capuz-b...,https://www.adidas.com.br/agasalho-com-capuz-b...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,BR,api,api_exact
4,2026-03-09,BR,AGASALHO COM CAPUZ BIG LOGO,KSG46,BRL,429.99,549.99,M,3GG,Lifestyle,...,-99.0,NaN,https://www.adidas.com.br/agasalho-com-capuz-b...,https://www.adidas.com.br/agasalho-com-capuz-b...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,BR,api,api_exact
5,2026-03-09,BR,AGASALHO COM CAPUZ BIG LOGO,KSG46,BRL,429.99,549.99,M,3GG/T,Lifestyle,...,-99.0,NaN,https://www.adidas.com.br/agasalho-com-capuz-b...,https://www.adidas.com.br/agasalho-com-capuz-b...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,BR,api,api_exact
6,2026-03-09,BR,AGASALHO COM CAPUZ BIG LOGO,KSG46,BRL,429.99,549.99,M,3XT2,Lifestyle,...,-99.0,NaN,https://www.adidas.com.br/agasalho-com-capuz-b...,https://www.adidas.com.br/agasalho-com-capuz-b...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,BR,api,api_exact
7,2026-03-09,BR,AGASALHO COM CAPUZ BIG LOGO,KSG46,BRL,429.99,549.99,M,3XT3,Lifestyle,...,-99.0,NaN,https://www.adidas.com.br/agasalho-com-capuz-b...,https://www.adidas.com.br/agasalho-com-capuz-b...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,BR,api,api_exact
8,2026-03-09,BR,AGASALHO COM CAPUZ BIG LOGO,KSG46,BRL,429.99,549.99,M,4GG,Lifestyle,...,-99.0,NaN,https://www.adidas.com.br/agasalho-com-capuz-b...,https://www.adidas.com.br/agasalho-com-capuz-b...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,BR,api,api_exact
9,2026-03-09,BR,AGASALHO COM CAPUZ BIG LOGO,KSG46,BRL,429.99,549.99,M,4GG/T,Lifestyle,...,-99.0,NaN,https://www.adidas.com.br/agasalho-com-capuz-b...,https://www.adidas.com.br/agasalho-com-capuz-b...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,BR,api,api_exact
